# 3. TextSplitter 源码与 CharacterTextSplitter

来源：《尚硅谷-10-RAG.pdf》，第 24–31 页；对应课件 **2.3.1–2.3.3、2.3.4 ①**。页码均指 PDF 阅读器中的页序。

整理方式：原理保留关键逻辑；第 2 章示例精简，重点学习拆分器源码与递归拆分。
只记录到 2.5.2，不收录 2.5.3 案例。

**运行约定**：在 `D:/grow/langchain/7_RAG` 打开本文件，选择
`D:/conda/envs/langchain/python.exe` 对应的 Python 内核；代码单元按本文件顺序运行。
各 Notebook 的变量相互独立。使用 `len` 的示例以 Python 字符长度计数，不能据此推断 Token 数。

## 3.1 分块先理解这几个参数

| 参数 | 基类默认值 | 要点 |
| --- | --- | --- |
| `chunk_size` | `4000` | 目标块长上限；能否严格限制还取决于子类是否能继续细分 |
| `chunk_overlap` | `200` | 合并时希望保留的重叠窗口长度，不是每对相邻块的精确重叠承诺 |
| `length_function` | `len` | 决定长度单位；Python 字符数与 Token 数不同 |
| `keep_separator` | `False` | 是否保留拆分符；`True` 等价于 `"start"`，也可指定 `"end"` |
| `add_start_index` | `False` | 在生成 Document 时写入 `metadata["start_index"]` |
| `strip_whitespace` | `True` | 清理最终块两端空白 |

`TextSplitter` 是抽象基类，不能直接实例化。它负责公共流程，子类实现 `split_text()`。

## 3.2 三个入口与调用链

| 方法 | 输入 | 输出 | 常见用法 |
| --- | --- | --- | --- |
| `split_text(text)` | `str` | `list[str]` | 只关心文本块 |
| `create_documents(texts, metadatas)` | `list[str]` 和可选元数据 | `list[Document]` | 已有字符串，需要封装文档 |
| `split_documents(docs)` | `Iterable[Document]` | `list[Document]` | 接 Loader 的结果，延续原来的元数据 |

```text
split_documents → create_documents → 子类.split_text
                        ↓
              deepcopy(metadata) → 新 Document

子类.split_text → 先拆成片段 → _merge_splits → 最终字符串块
```

`add_start_index=True` 对 `split_text()` 返回的字符串列表没有可附加的位置；
要观察这个元数据，需要 `create_documents()` 或 `split_documents()`。

## 3.3 源码阅读约定

以下节选自本地 **`langchain-text-splitters==1.1.1`** 的
`D:/conda/envs/langchain/Lib/site-packages/langchain_text_splitters/base.py`。
原作者为 LangChain 项目，按其 MIT 许可证分发。
**仅省略英文 docstring 和原英文整行注释，添加中文注释；执行语句保持原样。**
每个代码框是类内方法节选，供阅读，不是一个可单独运行的完整类；实际示例调用安装包。
Tokenizer 工厂等非本次重点没有全文展开。

### 3.3.1 __init__：校验与保存公共参数

```python
def __init__(
    self,
    chunk_size: int = 4000,
    chunk_overlap: int = 200,
    length_function: Callable[[str], int] = len,
    keep_separator: bool | Literal["start", "end"] = False,  # noqa: FBT001,FBT002
    add_start_index: bool = False,  # noqa: FBT001,FBT002
    strip_whitespace: bool = True,  # noqa: FBT001,FBT002
) -> None:
    # 先校验长度参数；单位由 length_function 决定，不一定是字符。
    if chunk_size <= 0:
        msg = f"chunk_size must be > 0, got {chunk_size}"
        raise ValueError(msg)
    if chunk_overlap < 0:
        msg = f"chunk_overlap must be >= 0, got {chunk_overlap}"
        raise ValueError(msg)
    # 实现允许 overlap == size；实际使用通常应取更小值。
    if chunk_overlap > chunk_size:
        msg = (
            f"Got a larger chunk overlap ({chunk_overlap}) than chunk size "
            f"({chunk_size}), should be smaller."
        )
        raise ValueError(msg)
    # 保存配置；基类负责公共规则，具体拆分由子类 split_text 完成。
    self._chunk_size = chunk_size
    self._chunk_overlap = chunk_overlap
    self._length_function = length_function
    self._keep_separator = keep_separator
    self._add_start_index = add_start_index
    self._strip_whitespace = strip_whitespace
```

### 3.3.2 split_documents：取出原文和元数据

```python
def split_documents(self, documents: Iterable[Document]) -> list[Document]:
    # 先把 Document 拆成平行的正文列表和元数据列表。
    texts, metadatas = [], []
    for doc in documents:
        texts.append(doc.page_content)
        metadatas.append(doc.metadata)
    # 统一进入 create_documents，再由子类 split_text 执行切分。
    return self.create_documents(texts, metadatas=metadatas)
```

### 3.3.3 create_documents：调用子类并重新封装

```python
def create_documents(
    self, texts: list[str], metadatas: list[dict[Any, Any]] | None = None
) -> list[Document]:
    metadatas_ = metadatas or [{}] * len(texts)
    documents = []
    for i, text in enumerate(texts):
        index = 0
        previous_chunk_len = 0
        # 多态调用：Character、Recursive 等子类决定如何生成字符串块。
        for chunk in self.split_text(text):
            # 每个块深拷贝一份 metadata，避免块之间共享可变字典。
            metadata = copy.deepcopy(metadatas_[i])
            if self._add_start_index:
                # 从上块末尾减去 overlap 处开始找；这是源码的定位策略。
                offset = index + previous_chunk_len - self._chunk_overlap
                # 返回的是当前输入 text 中的字符下标，不是 Token 下标。
                index = text.find(chunk, max(0, offset))
                metadata["start_index"] = index
                previous_chunk_len = len(chunk)
            # 将字符串块重新封装成 Document。
            new_doc = Document(page_content=chunk, metadata=metadata)
            documents.append(new_doc)
    return documents
```

`start_index` 相对于传入的那一份 `page_content`；PDF 按页加载时就是页内位置。
它通过 `find()` 定位：遇到复杂重复文本，或使用 Token 长度时，不应把它当作适用于任何情况的精确溯源算法。

### 3.3.4 _join_docs 与 _merge_splits：合并和重叠的关键

```python
def _join_docs(self, docs: list[str], separator: str) -> str | None:
    text = separator.join(docs)
    if self._strip_whitespace:
        # 默认移除块两端空白，所以输出长度可能小于合并时累计的长度。
        text = text.strip()
    return text or None
```

```python
def _merge_splits(self, splits: Iterable[str], separator: str) -> list[str]:
    # 拼接用的分隔符也占长度；current_doc 是候选窗口，total 是其累计长度。
    separator_len = self._length_function(separator)

    docs = []
    current_doc: list[str] = []
    total = 0
    for d in splits:
        len_ = self._length_function(d)
        if (
            total + len_ + (separator_len if len(current_doc) > 0 else 0)
            > self._chunk_size
        ):
            # 若单个原始片段已经超长，此函数只警告，不会递归拆开它。
            if total > self._chunk_size:
                logger.warning(
                    "Created a chunk of size %d, which is longer than the "
                    "specified %d",
                    total,
                    self._chunk_size,
                )
            if len(current_doc) > 0:
                # 输出当前候选窗口；循环结束后的调用用于收尾。
                doc = self._join_docs(current_doc, separator)
                if doc is not None:
                    docs.append(doc)
                # 从左侧弹出旧片段，直到剩余长度不大于 overlap，且能容纳下一片段。
                while total > self._chunk_overlap or (
                    total + len_ + (separator_len if len(current_doc) > 0 else 0)
                    > self._chunk_size
                    and total > 0
                ):
                    total -= self._length_function(current_doc[0]) + (
                        separator_len if len(current_doc) > 1 else 0
                    )
                    # 按整个片段移除；不能只保留半个片段，所以实际重叠可能少于设置值。
                    current_doc = current_doc[1:]
        # 把本轮片段加入窗口，然后继续尝试合并后续片段。
        current_doc.append(d)
        total += len_ + (separator_len if len(current_doc) > 1 else 0)
    # 输出当前候选窗口；循环结束后的调用用于收尾。
    doc = self._join_docs(current_doc, separator)
    if doc is not None:
        docs.append(doc)
    return docs
```

### 用一组数字看合并过程

假设原始片段是 `['aaa', 'bbb', 'ccc']`，拼接符为空，`chunk_size=6`：

| overlap | 第一个块 | 加入 ccc 前保留什么 | 最后结果 |
| --- | --- | --- | --- |
| `3` | `aaabbb` | `bbb` 长度为 3，可以保留 | `['aaabbb', 'bbbccc']` |
| `2` | `aaabbb` | 不能只保留半个 `bbb`，两个旧片段都移除 | `['aaabbb', 'ccc']` |

所以 `chunk_overlap=2` 也可能实际重叠 0 个字符。另一个原因是递归拆分的不同分支分别合并，
分支之间不会强行补足重叠。

In [9]:
from langchain_text_splitters import CharacterTextSplitter

# 直接调用受保护方法，仅用于观察源码；实际业务用公共入口。
for overlap in (3, 2):
    splitter = CharacterTextSplitter(chunk_size=6, chunk_overlap=overlap)
    print(f"overlap={overlap}:", splitter._merge_splits(["aaa", "bbb", "ccc"], ""))

overlap=3: ['aaabbb', 'bbbccc']
overlap=2: ['aaabbb', 'ccc']


## 3.4 CharacterTextSplitter：一个分隔符，先拆后合并

核心步骤：将 `separator` 转成拆分模式 → 得到片段列表 → `_merge_splits()`。
默认 `separator="\n\n"`。它不会在一个超长片段上自动尝试更多分隔符。
因此 `chunk_size` 在这里可能被超出；警告也不是在每一种超长情况下一定输出。

`separator=""` 表示先按单个字符细分，再合并成块。
它不是「完全不拆分」。下面只保留一个对比实例。

In [32]:
from langchain_text_splitters import CharacterTextSplitter

text = "这是第一段文本。这是第二段内容。最后一段结束。"

for separator in ("。", ""):
    splitter = CharacterTextSplitter(
        separator=separator,
        chunk_size=8,
        chunk_overlap=0,
        keep_separator="end",  # 句号放在前一片段的末尾
    )
    chunks = splitter.split_text(text)
    print("separator =", repr(separator))
    for chunk in chunks:
        print(len(chunk), repr(chunk))

# 没有双换行，整段会保留；不能把 chunk_size 当作这里的硬切刀。
splitter = CharacterTextSplitter(chunk_size=10, chunk_overlap=0)
chunks = splitter.split_text(text)
print("无匹配分隔符，块长:", [len(chunk) for chunk in chunks])

separator = '。'
8 '这是第一段文本。'
8 '这是第二段内容。'
7 '最后一段结束。'
separator = ''
8 '这是第一段文本。'
8 '这是第二段内容。'
7 '最后一段结束。'
无匹配分隔符，块长: [23]


## 3.5 Document 入口：保留来源，新增位置

In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter

docs = [Document(page_content="ABCDEFGHIJKLMNO", metadata={"source": "demo.txt", "page": 0})]
splitter = CharacterTextSplitter(
    separator="", chunk_size=6, chunk_overlap=2, add_start_index=True
)
paragraphs = splitter.split_documents(docs)
for para in paragraphs:
    print(para.page_content, para.metadata)

# 只修改一个块的元数据，原文档不会跟着修改。
paragraphs[0].metadata["tag"] = "first"
print("原文档:", docs[0].metadata)
print("另一个块:", paragraphs[1].metadata)

ABCDEF {'source': 'demo.txt', 'page': 0, 'start_index': 0}
EFGHIJ {'source': 'demo.txt', 'page': 0, 'start_index': 4}
IJKLMN {'source': 'demo.txt', 'page': 0, 'start_index': 8}
MNO {'source': 'demo.txt', 'page': 0, 'start_index': 12}
原文档: {'source': 'demo.txt', 'page': 0}
另一个块: {'source': 'demo.txt', 'page': 0, 'start_index': 4}


**复习要点**：入口区分、子类多态、元数据深拷贝、合并窗口、重叠不是保证值。
普通长文本通常继续学习下一份的递归拆分器。

## 笔记导航

1. [7_1_RAG原理与流程](7_1_RAG原理与流程.ipynb)
2. [7_2_Document与文档加载](7_2_Document与文档加载.ipynb)
3. [7_3_TextSplitter源码与Character切分](7_3_TextSplitter源码与Character切分.ipynb)
4. [7_4_RecursiveCharacterTextSplitter详解](7_4_RecursiveCharacterTextSplitter详解.ipynb)
5. [7_5_其他拆分方式速记](7_5_其他拆分方式速记.ipynb)
6. [7_6_Embedding与向量存储](7_6_Embedding与向量存储.ipynb)